In [1]:
import torch


# Check if CUDA (NVIDIA GPU) is available
is_cuda = torch.cuda.is_available()
print(f"GPU Available (CUDA): {is_cuda}")

if is_cuda:
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


GPU Available (CUDA): True
Number of GPUs: 1
GPU Name: Tesla T4


In [2]:
!pip install -q "pydantic-ai-slim[openai]" fastmcp nest_asyncio openai fastapi uvicorn requests pyjwt python-dotenv python-multipart python-weather aiohttp beautifulsoup4  huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.2/876.2 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.5/279.5 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.1/148.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17

In [3]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (382 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directo

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
import nest_asyncio; nest_asyncio.apply()      # lets `await` work inside a notebook

from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

def ollama_model(name):
    # Ollama copies the OpenAI API shape, so we reuse the OpenAI client and swap the URL.
    return OpenAIChatModel(
        name,
        provider=OpenAIProvider(base_url="http://localhost:11434/v1", api_key="ollama"),
    )

model = ollama_model("gemma4:e4b")
print("setup done")

setup done


#Session 1 - JSON RPC 2.0

In [8]:
request = {
    "jsonrpc": "2.0",
    id: 1,
    "method": "add",
    "params": {
        "a": 2,
        "b": 3
    }
}

Response Result or Error

In [9]:
ok  = {"jsonrpc": "2.0", "id": 1, "result": 5}
bad = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}

In [10]:
def add(a, b):
  return a + b
def shout(text):
  return text.upper() + "!"

FUNCTIONS = {"add": add, "shout": shout}      # name (text) -> real Python function


def handle(req):
    fn = FUNCTIONS.get(req["method"])                       # look up by NAME

    if fn is None:                                          # no such function -> error reply
        return {"jsonrpc": "2.0", "id": req["id"],
                "error": {"code": -32601, "message": "Method not found"}}

    result = fn(**req["params"])                            # <-- THE REMOTE CALL.
                                                            #     ** turns {"a":2,"b":3} into add(a=2, b=3)
    return {"jsonrpc": "2.0", "id": req["id"], "result": result}


print(handle({"jsonrpc": "2.0", "id": 1, "method": "add",   "params": {"a": 40, "b": 2}}))
print(handle({"jsonrpc": "2.0", "id": 2, "method": "shout", "params": {"text": "hello"}}))
print(handle({"jsonrpc": "2.0", "id": 3, "method": "fly",   "params": {}}))

{'jsonrpc': '2.0', 'id': 1, 'result': 42}
{'jsonrpc': '2.0', 'id': 2, 'result': 'HELLO!'}
{'jsonrpc': '2.0', 'id': 3, 'error': {'code': -32601, 'message': 'Method not found'}}


In [11]:
def add(a, b):
  return a + b
def shout(text):
  return text.upper() + "!"

FUNCTIONS = {"add": add, "shout": shout}      # name (text) -> real Python function


TOOLS = [
    {
        "name": "add",                                                   # matches FUNCTIONS["add"]
        "description": "Add two numbers together and return the sum.",   # the LLM reads this
        "inputSchema": {
            "type": "object",                        # arguments always arrive as a dict
            "properties": {"a": {"type": "number"},  # argument a -> a number
                           "b": {"type": "number"}}, # argument b -> a number
            "required": ["a", "b"],                  # both mandatory
        },
    },
    {
        "name": "shout",
        "description": "Return the given text in capital letters with an exclamation mark.",
        "inputSchema": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
        },
    },
]